# Exploring Zigzag Persistence on Neural Response Grids

This notebook explores zigzag persistence diagrams from 3D neural activity grids
(shape `15×15×10×n_frames`) recorded during video stimuli presentation.

**Goals:**
1. Load and inspect trial data from multiple mice
2. Understand the activation distribution and compute percentile-based thresholds
3. Run zigzag persistence across a range of `p_active` percentiles (10–50)
4. Visualize and compare persistence diagrams across thresholds
5. Assess stability and informativeness to select the optimal threshold
6. (Bonus) Compare with static per-frame GPU persistence
7. Check cross-mouse consistency of the chosen threshold

In [ ]:
import os
import glob
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

from zztop import (
    run_cubical_zigzag,
    run_cubical_persistence_gpu,
    gpu_available,
    gpu_supports_3d,
)

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## 1. Data loading and inspection

In [ ]:
DATA_ROOT = Path(
    "/orfeo/scratch/area/ygardinazzi/sensorium_2026/derivatives/"
    "grid-15x15x10_norm-by_minmax"
)

# List all mice
mice = sorted([d.name for d in DATA_ROOT.iterdir() if d.is_dir()])
print(f"Found {len(mice)} mice:")
for m in mice:
    n_trials = len(list((DATA_ROOT / m / "trials").glob("*.npy")))
    print(f"  {m}: {n_trials} trials")

In [ ]:
# Pick a reference mouse and load a handful of trials
REF_MOUSE = mice[0]
trial_files = sorted((DATA_ROOT / REF_MOUSE / "trials").glob("*.npy"))

# Sample ~8 trials spread across the list
rng = np.random.default_rng(42)
N_SAMPLE = 8
sample_idx = np.linspace(0, len(trial_files) - 1, N_SAMPLE, dtype=int)
sample_files = [trial_files[i] for i in sample_idx]

sample_data = {}
for f in sample_files:
    data = np.load(f)
    trial_name = f.stem.split("_trial-")[-1]
    sample_data[f"trial-{trial_name}"] = data
    print(f"  {f.name}: shape={data.shape}, range=[{data.min():.4f}, {data.max():.4f}]")

print(f"\nLoaded {len(sample_data)} trials from mouse: {REF_MOUSE}")

## 2. Activation distribution and threshold computation

The `threshold` parameter in `run_cubical_zigzag` controls which higher-dimensional
cubical cells are "active" at each frame. Internally, gird values are **negated**
before being passed to GUDHI, so a cell with activation $a$ gets filtration value $-a$.
A cell is active if its filtration value $< \text{threshold}$, i.e. if $-a < \text{threshold}$,
i.e. if $a > -\text{threshold}$.

We define `p_active` as a percentile of the positive activation values across all
voxels and frames in a trial. Higher `p_active` → lower threshold → more cells active
→ richer topology.

In [ ]:
def compute_threshold(data, p_active):
    """Compute the activation threshold for a given p_active percentile.
    
    The threshold is the negative of the p_active-th percentile of positive
    activation values. This matches zz-top's internal convention where grid
    values are negated.
    
    Parameters
    ----------
    data : np.ndarray
        Grid data of shape (N1, ..., Nd, n_frames).
    p_active : float
        Percentile (0-100) of positive activations to use as threshold.
        Higher p_active → more cells active → richer topology.
    
    Returns
    -------
    float
        Threshold value to pass to run_cubical_zigzag.
    """
    positive_vals = data[data > 0].ravel()
    if len(positive_vals) == 0:
        return 0.0
    activation_cutoff = np.percentile(positive_vals, p_active)
    return -activation_cutoff  # Negate to match zz-top convention

In [ ]:
# Visualize the activation distribution for a few trials
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
p_active_values = [10, 20, 30, 40, 50]

for ax, (name, data) in zip(axes.flat, sample_data.items()):
    pos_vals = data[data > 0].ravel()
    ax.hist(pos_vals, bins=100, density=True, alpha=0.7, color="steelblue")
    # Mark percentile thresholds
    for p in p_active_values:
        pval = np.percentile(pos_vals, p)
        ax.axvline(pval, color="red", linestyle="--", alpha=0.6, linewidth=1)
        ax.text(pval, ax.get_ylim()[1] * 0.9, f"p{p}", fontsize=7, color="red",
                ha="center", rotation=45)
    ax.set_title(name, fontsize=9)
    ax.set_xlabel("Activation")

fig.suptitle(f"Activation distributions — {REF_MOUSE.split('-')[0]}", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Compute thresholds for all sampled trials at each p_active
threshold_table = {}
for p in p_active_values:
    thresholds = []
    for name, data in sample_data.items():
        t = compute_threshold(data, p)
        thresholds.append(t)
    threshold_table[p] = thresholds
    mean_t = np.mean(thresholds)
    std_t = np.std(thresholds)
    print(f"  p_active={p:3d}: threshold = {mean_t:.6f} ± {std_t:.6f}")

print("\nNote: threshold is negative (matches negated GUDHI convention).")
print("A more negative threshold → higher activation cutoff → fewer active cells.")

## 3. Zigzag persistence across thresholds

We run `run_cubical_zigzag` for a subset of trials at each `p_active` value.
This is the core experiment: how does the choice of threshold affect the
resulting persistence diagrams?

In [ ]:
# Run zigzag persistence for each trial × each p_active
# Use only 3 trials to keep compute time reasonable
N_ZZ_TRIALS = 3
zz_trial_names = list(sample_data.keys())[:N_ZZ_TRIALS]

# run_cubical_zigzag returns (dim, birth_frame, death_frame) directly
zz_results = {}  # zz_results[p_active][trial_name] = bars
zz_nframes = {}  # zz_nframes[trial_name] = n_frames

for p in p_active_values:
    zz_results[p] = {}
    for name in zz_trial_names:
        data = sample_data[name]
        threshold = compute_threshold(data, p)
        nf = data.shape[-1]
        zz_nframes[name] = nf
        
        t0 = time.time()
        bars = run_cubical_zigzag(data, threshold=threshold)
        elapsed = time.time() - t0
        
        zz_results[p][name] = bars
        
        # Count bars per dimension
        dims = [b[0] for b in bars]
        for d in sorted(set(dims)):
            n_d = sum(1 for x in dims if x == d)
            print(f"  p_active={p}, {name}, dim {d}: {n_d} bars")
        print(f"    → {len(bars)} total bars, {nf} frames, computed in {elapsed:.1f}s")
    print()

## 4. Visualize persistence diagrams

In [ ]:
def plot_persistence_diagram(bars, ax, title="", max_dim=2, n_frames=None):
    """Plot a persistence diagram (birth vs death) from zigzag barcode.
    
    bars should contain frame-indexed birth/death values.
    """
    colors = {0: "tab:blue", 1: "tab:orange", 2: "tab:green"}
    labels = {0: "H₀ (components)", 1: "H₁ (loops)", 2: "H₂ (voids)"}
    
    for dim in range(max_dim + 1):
        pts = [(b, d) for (dd, b, d) in bars if dd == dim and d != float("inf")]
        if pts:
            births, deaths = zip(*pts)
            ax.scatter(births, deaths, s=8, alpha=0.5, c=colors.get(dim, "gray"),
                      label=f"{labels.get(dim, f'H_{dim}')} ({len(pts)})")
    
    # Diagonal line
    max_val = n_frames if n_frames else max(ax.get_xlim()[1], ax.get_ylim()[1], 1)
    ax.plot([0, max_val], [0, max_val], "k--", alpha=0.3, lw=0.8)
    if n_frames:
        ax.set_xlim(-1, n_frames)
        ax.set_ylim(-1, n_frames)
    ax.set_xlabel("Birth (frame)")
    ax.set_ylabel("Death (frame)")
    ax.set_title(title, fontsize=9)
    ax.legend(fontsize=7, loc="lower right")
    ax.set_aspect("equal")

In [ ]:
# Persistence diagrams: one row per trial, one column per p_active
fig, axes = plt.subplots(
    N_ZZ_TRIALS, len(p_active_values),
    figsize=(4 * len(p_active_values), 4 * N_ZZ_TRIALS),
    squeeze=False,
)

for i, name in enumerate(zz_trial_names):
    nf = zz_nframes[name]
    for j, p in enumerate(p_active_values):
        bars = zz_results[p][name]
        plot_persistence_diagram(bars, axes[i, j],
                                 title=f"{name}, p_active={p}", n_frames=nf)

fig.suptitle("Zigzag persistence diagrams across thresholds", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
def plot_persistence_barcode(bars, ax, title="", max_dim=2, n_frames=None):
    """Plot a persistence barcode from zigzag barcode (frame-indexed)."""
    colors = {0: "tab:blue", 1: "tab:orange", 2: "tab:green"}
    
    y_offset = 0
    yticks, ytick_labels = [], []
    
    for dim in range(max_dim + 1):
        dim_bars = [(b, d) for (dd, b, d) in bars if dd == dim and d != float("inf")]
        # Sort by persistence (longest first)
        dim_bars.sort(key=lambda x: -(x[1] - x[0]))
        
        start_y = y_offset
        for b, d in dim_bars:
            ax.plot([b, d], [y_offset, y_offset], color=colors.get(dim, "gray"),
                   linewidth=0.8, alpha=0.7)
            y_offset += 1
        if dim_bars:
            mid = (start_y + y_offset) / 2
            yticks.append(mid)
            ytick_labels.append(f"H{dim} ({len(dim_bars)})")
        y_offset += 2  # gap between dimensions
    
    ax.set_yticks(yticks)
    ax.set_yticklabels(ytick_labels, fontsize=8)
    ax.set_xlabel("Frame")
    if n_frames:
        ax.set_xlim(-1, n_frames)
    ax.set_title(title, fontsize=9)

In [ ]:
# Barcode visualization for one trial across thresholds
ref_trial = zz_trial_names[0]
nf = zz_nframes[ref_trial]
fig, axes = plt.subplots(len(p_active_values), 1,
                          figsize=(14, 3 * len(p_active_values)))

for ax, p in zip(axes, p_active_values):
    bars = zz_results[p][ref_trial]
    plot_persistence_barcode(bars, ax, title=f"{ref_trial}, p_active={p}",
                             n_frames=nf)

fig.suptitle("Zigzag barcodes — effect of threshold", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5. Stability analysis: choosing the optimal threshold

We now quantify how diagram properties change with `p_active` to find the
"sweet spot" — the threshold that produces informative, stable, and
discriminative diagrams.

In [ ]:
def barcode_stats(bars, max_dim=2):
    """Compute summary statistics from a zigzag barcode."""
    stats = {}
    for dim in range(max_dim + 1):
        finite_bars = [(b, d) for (dd, b, d) in bars if dd == dim and d != float("inf")]
        if not finite_bars:
            stats[dim] = {
                "n_bars": 0, "mean_pers": 0, "median_pers": 0,
                "total_pers": 0, "max_pers": 0, "std_pers": 0,
            }
            continue
        persistences = [d - b for b, d in finite_bars]
        stats[dim] = {
            "n_bars": len(finite_bars),
            "mean_pers": np.mean(persistences),
            "median_pers": np.median(persistences),
            "total_pers": np.sum(persistences),
            "max_pers": np.max(persistences),
            "std_pers": np.std(persistences),
        }
    return stats

In [ ]:
# Collect statistics across trials and thresholds
all_stats = {}  # all_stats[p_active] = list of stats dicts (one per trial)

for p in p_active_values:
    all_stats[p] = []
    for name in zz_trial_names:
        bars = zz_results[p][name]
        s = barcode_stats(bars)
        all_stats[p].append(s)

# Plot summary statistics as a function of p_active
metrics = ["n_bars", "mean_pers", "total_pers", "max_pers"]
metric_labels = ["# Bars", "Mean persistence", "Total persistence", "Max persistence"]

fig, axes = plt.subplots(len(metrics), 3, figsize=(14, 3 * len(metrics)), sharey="row")
dim_labels = ["H₀", "H₁", "H₂"]

for row, (metric, mlabel) in enumerate(zip(metrics, metric_labels)):
    for dim in range(3):
        ax = axes[row, dim]
        for t_idx, name in enumerate(zz_trial_names):
            values = [all_stats[p][t_idx][dim][metric] for p in p_active_values]
            ax.plot(p_active_values, values, "o-", alpha=0.7, label=name, markersize=4)
        ax.set_xlabel("p_active")
        if dim == 0:
            ax.set_ylabel(mlabel)
        ax.set_title(f"{dim_labels[dim]}")
        if row == 0 and dim == 0:
            ax.legend(fontsize=7)

fig.suptitle("Diagram statistics vs. threshold percentile", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Coefficient of variation (CV) across trials — measures inter-trial stability
print("Coefficient of Variation (CV) across trials — lower = more stable:\n")
print(f"{'p_active':>10} | {'Metric':>18} | {'H0 CV':>8} | {'H1 CV':>8} | {'H2 CV':>8}")
print("-" * 70)

for p in p_active_values:
    for metric in ["n_bars", "total_pers"]:
        cvs = []
        for dim in range(3):
            vals = [all_stats[p][t_idx][dim][metric] for t_idx in range(N_ZZ_TRIALS)]
            mean_v = np.mean(vals)
            std_v = np.std(vals)
            cv = std_v / mean_v if mean_v > 0 else float("nan")
            cvs.append(cv)
        print(f"{p:>10} | {metric:>18} | {cvs[0]:>8.3f} | {cvs[1]:>8.3f} | {cvs[2]:>8.3f}")
    print()

## 6. Cross-mouse consistency

Check whether the optimal `p_active` generalizes across mice, or whether
each mouse needs its own threshold.

In [ ]:
# Load one trial from each of 3 different mice
cross_mouse_data = {}
cross_mouse_names = [mice[0], mice[4], mice[8]]  # Spread across the list

for m in cross_mouse_names:
    trials = sorted((DATA_ROOT / m / "trials").glob("*.npy"))
    # Pick a trial in the middle
    trial_file = trials[len(trials) // 2]
    data = np.load(trial_file)
    short_name = m.split("-")[0] + m.split("-")[1]
    cross_mouse_data[short_name] = data
    print(f"  {m[:30]}...: shape={data.shape}")

In [ ]:
# Run zigzag persistence at each p_active for each mouse
cross_mouse_results = {}  # cross_mouse_results[mouse_name][p_active] = bars (frame-indexed)

for mname, data in cross_mouse_data.items():
    cross_mouse_results[mname] = {}
    nf = data.shape[-1]
    for p in p_active_values:
        threshold = compute_threshold(data, p)
        t0 = time.time()
        bars = run_cubical_zigzag(data, threshold=threshold)
        elapsed = time.time() - t0
        cross_mouse_results[mname][p] = bars
        n_total = len([b for b in bars if b[2] != float("inf")])
        print(f"  {mname}, p_active={p}: {n_total} finite bars, {nf} frames ({elapsed:.1f}s)")
    print()

In [ ]:
# Compare cross-mouse: total persistence and bar counts across p_active
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for mname in cross_mouse_results:
    for dim in range(3):
        n_bars_list = []
        total_pers_list = []
        for p in p_active_values:
            bars = cross_mouse_results[mname][p]
            s = barcode_stats(bars)
            n_bars_list.append(s[dim]["n_bars"])
            total_pers_list.append(s[dim]["total_pers"])
        axes[0, dim].plot(p_active_values, n_bars_list, "o-", label=mname, markersize=5)
        axes[1, dim].plot(p_active_values, total_pers_list, "o-", label=mname, markersize=5)

for dim in range(3):
    axes[0, dim].set_title(f"H{dim} — # Bars")
    axes[1, dim].set_title(f"H{dim} — Total persistence")
    axes[0, dim].set_xlabel("p_active")
    axes[1, dim].set_xlabel("p_active")

axes[0, 0].legend(fontsize=8)
fig.suptitle("Cross-mouse comparison of zigzag persistence", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Static per-frame GPU persistence (comparison)

For comparison, run standard cubical persistence frame-by-frame on one trial
using the GPU backend. This shows what *within-frame* topology looks like,
without the temporal zigzag structure.

In [ ]:
print(f"GPU available: {gpu_available()}")
print(f"GPU supports 3D: {gpu_supports_3d()}")

In [ ]:
# Run static per-frame persistence on one trial
ref_name = zz_trial_names[0]
ref_data = sample_data[ref_name]

backend = "gpu" if gpu_available() else "cpu"
print(f"Running static per-frame persistence with backend='{backend}'...")
t0 = time.time()
static_diagrams = run_cubical_persistence_gpu(ref_data, backend=backend)
elapsed = time.time() - t0
print(f"Done in {elapsed:.1f}s — {len(static_diagrams)} frames")

In [ ]:
# Plot a few frames' static persistence diagrams
n_frames = ref_data.shape[-1]
frame_indices = np.linspace(0, n_frames - 1, 6, dtype=int)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
colors = {0: "tab:blue", 1: "tab:orange", 2: "tab:green"}

for ax, fi in zip(axes.flat, frame_indices):
    diag = static_diagrams[fi]
    for dim in sorted(diag.keys()):
        pts = diag[dim]
        finite_mask = np.isfinite(pts[:, 1])
        pts_f = pts[finite_mask]
        if len(pts_f) > 0:
            ax.scatter(pts_f[:, 0], pts_f[:, 1], s=10, alpha=0.5,
                      c=colors.get(dim, "gray"), label=f"H{dim} ({len(pts_f)})")
    lims = ax.get_xlim()
    ax.plot([min(lims), max(lims)], [min(lims), max(lims)], "k--", alpha=0.3, lw=0.8)
    ax.set_xlabel("Birth")
    ax.set_ylabel("Death")
    ax.set_title(f"Frame {fi}", fontsize=10)
    ax.legend(fontsize=7)

fig.suptitle(f"Static per-frame persistence — {ref_name}", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Track how Betti numbers evolve across frames in static persistence
betti_over_time = {dim: [] for dim in range(3)}

for t in range(n_frames):
    diag = static_diagrams[t]
    for dim in range(3):
        if dim in diag:
            # Count finite bars (alive features)
            n_finite = np.sum(np.isfinite(diag[dim][:, 1]))
            betti_over_time[dim].append(n_finite)
        else:
            betti_over_time[dim].append(0)

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
dim_labels = ["H₀ (components)", "H₁ (loops)", "H₂ (voids)"]

for dim, ax in enumerate(axes):
    ax.plot(betti_over_time[dim], color=colors[dim], linewidth=0.8)
    ax.set_ylabel(f"# {dim_labels[dim]}")
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Frame")
fig.suptitle(f"Per-frame Betti numbers (static persistence) — {ref_name}", fontsize=14)
plt.tight_layout()
plt.show()

## 8. Conclusion: threshold selection

Based on the above analysis, choose the optimal `p_active` value.

In [ ]:
# ============================================================
# DECISION: Set the chosen p_active here based on the analysis
# ============================================================
CHOSEN_P_ACTIVE = 30  # <-- UPDATE after inspecting the plots above

# Compute per-mouse thresholds at the chosen p_active
print(f"Chosen p_active = {CHOSEN_P_ACTIVE}\n")
print("Per-mouse thresholds (computed from one sampled trial each):")
print("=" * 80)

mouse_thresholds = {}
for m in mice:
    trials = sorted((DATA_ROOT / m / "trials").glob("*.npy"))
    # Sample a few trials to get a stable threshold estimate
    sample_indices = np.linspace(0, len(trials) - 1, min(10, len(trials)), dtype=int)
    thresholds = []
    for idx in sample_indices:
        data = np.load(trials[idx])
        t = compute_threshold(data, CHOSEN_P_ACTIVE)
        thresholds.append(t)
    mean_thresh = np.mean(thresholds)
    mouse_thresholds[m] = mean_thresh
    print(f"  {m[:40]}...: threshold = {mean_thresh:.6f} (± {np.std(thresholds):.6f})")

print(f"\nGlobal mean threshold: {np.mean(list(mouse_thresholds.values())):.6f}")
print(f"Global std threshold:  {np.std(list(mouse_thresholds.values())):.6f}")
print("\nRecommendation: Use per-mouse thresholds computed from a sample of trials.")